## DQN Labs AI - **dqnCode** runtime

## ⬇ Run this:

In [ ]:
#@title #Runtime Info
gpu_info = !nvidia-smi
gpu_info = '\n'.join(gpu_info)
if gpu_info.find('failed') >= 0:
  print('Not connected to a GPU')
else:
  print(gpu_info)
from psutil import virtual_memory
ram_gb = virtual_memory().total / 1e9
print('Your runtime has {:.1f} gigabytes of available RAM\n'.format(ram_gb))
if ram_gb < 20:
  print('Not using a high-RAM runtime')
else:
  print('You are using a high-RAM runtime!')


In [ ]:
#@title ⚡ llama cpp download and  (dqnGPT)

%cd /content/

!rm -rf build llama_bin.tar.gz *.gguf
print("removed old build")

!wget -q https://huggingface.co/DQN-Labs/llamacpp-binaries-for-colab/resolve/main/llama_bin.tar.gz
print("got new build")

!tar -xzf llama_bin.tar.gz
print("extracted new build")

!ls build/bin | head -5

In [ ]:
!wget https://huggingface.co/DQN-Labs/dqnCode-v0.2-1.5B/resolve/main/DQN-Code-v0.2-1.5B.Q4_K_M.gguf

In [ ]:
#@title CLOUDFLARED

!wget https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared
!pip install subprocess

In [ ]:
import re
import subprocess

# Run cloudflared tunnel in background and get the public URL
cloudflared_proc = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://127.0.0.1:8000', '--no-autoupdate'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True
)

public_url = None
for line in cloudflared_proc.stdout:
    print(line.strip())
    match = re.search(r'(https://.*\.trycloudflare\.com)', line)
    if match:
        public_url = match.group(1)
        break

if public_url:
    print(f"\n✅ Public URL for Ollama:\n{public_url}")
else:
    raise RuntimeError("❌ Could not find public Cloudflare URL.")


In [ ]:

import time
print("Please copy this address and paste it in the website, it will stay here for 10 seconds:")
print(public_url)
time.sleep(10)
!export LD_LIBRARY_PATH=/content/build/bin:$LD_LIBRARY_PATH && \
./build/bin/llama-server \
    -m DQN-Code-v0.2-1.5B.Q4_K_M.gguf \
    -ngl 999 \
    -c 16384 \
    --port 8000 --no-webui

## ⬇ Run this once the above is done.

In [ ]:
import time
print("Please copy this address and paste it in the website, it will stay here for 10 seconds:")
print(public_url)
time.sleep(10)
!export LD_LIBRARY_PATH=/content/build/bin:$LD_LIBRARY_PATH && \
./build/bin/llama-server \
    -m DQN-Code-v0.2-1.5B.Q4_K_M.gguf \
    -ngl 999 \
    -c 16384 \
    --port 8000 --no-webui